# Installing Dependencies

In [1]:
# Step 1: Installing all required packages
!pip install -q openai-whisper
!pip install -q pydub
!pip install -q openpyxl
!pip install -q jiwer
!apt-get install -q -y ffmpeg
!pip install openai-whisper

print("All dependencies installed.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 22.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 50.2 MB/s eta 0:00:00
Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 37 not upgraded.
All dependencies installed.


In [3]:
# first let's import everything we need
import whisper
import torch
import os
import re
import tempfile
import pandas as pd
from pydub import AudioSegment
from pydub.silence import detect_nonsilent
from openpyxl import load_workbook
from jiwer import wer
import warnings
warnings.filterwarnings('ignore')

# checking if GPU is available
# whisper large-v3 is a big model, GPU makes it way faster
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"using device: {device}")

if device == "cuda":
    print(f"GPU found: {torch.cuda.get_device_name(0)}, we're good to go!")
else:
    print("no GPU found")

/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


using device: cuda
GPU found: Tesla T4, we're good to go!


In [4]:
# mounting google drive so we can access our files
from google.colab import drive
drive.mount('/content/drive')

# setting the path to our AutoEIT folder
# change this if your folder name is different
FILES_DIR = "/content/drive/MyDrive/AutoEIT"

# let's verify all our files are there
print("files in our folder:")
for f in os.listdir(FILES_DIR):
    print(f"  {f}")

Mounted at /content/drive
files in our folder:
  038012_EIT-2A.mp3
  038011_EIT-1A.mp3
  038010_EIT-2A.mp3
  AutoEIT Sample Audio for Transcribing.xlsx
  038015_EIT-1A.mp3
  AutoEIT_Transcriptions_OUTPUT.xlsx


## Loading Whisper large-v3 model

In [5]:
# loading whisper large-v3 model
# large-v3 is the most accurate version, especially for spanish

model = whisper.load_model("large-v3", device=device)
print(f"model loaded successfully on {device}!")

100%|██████████████████████████████████████| 2.88G/2.88G [00:26<00:00, 117MiB/s]


model loaded successfully on cuda!


# Audio Transcription : Transcribing full audio with Whisper.




## Transcribing participant 038010-2A

In [6]:
# instead of manual segmentation, let's just give the whole audio to whisper.
# we can then pick out the 30 participant responses.

audio_path = os.path.join(FILES_DIR, "038010_EIT-2A.mp3")
audio = AudioSegment.from_mp3(audio_path)

# skip first 2:30 (english intro + practice)
audio_trimmed = audio[150000:]

# save trimmed audio temporarily
temp_path = "/tmp/038010_trimmed.wav"
audio_trimmed.export(temp_path, format="wav")

print("transcribing full audio with whisper.")

result = model.transcribe(
    temp_path,
    language="es",
    task="transcribe",
    temperature=0,
    condition_on_previous_text=False,
    word_timestamps=False,
    verbose=True   # shows output as it goes
)

print("\ndone!")

transcribing full audio with whisper.
[00:00.000 --> 00:03.000]  Repita lo máximo que puedas.
[00:04.000 --> 00:08.000]  Simplemente repita la frase hasta que oye el sonido.
[00:09.000 --> 00:10.000]  Ahora comencemos.
[00:11.000 --> 00:13.000]  Quiero cortarme el pelo.
[00:14.000 --> 00:15.000]  ¿Qué es eso?
[00:15.000 --> 00:17.000]  Quiero cortarme el pelo.
[00:18.000 --> 00:20.000]  El libro está en la mesa.
[00:22.000 --> 00:24.000]  El libro está en la mesa.
[00:26.000 --> 00:28.000]  El perro no tiene pelo.
[00:28.000 --> 00:33.000]  El carro lo tiene pero...
[00:33.000 --> 00:41.000]  Él se ducha cada mañana.
[00:41.000 --> 00:50.000]  ¿Qué dice usted que va a hacer hoy?
[00:50.000 --> 00:58.600]  Dudo que sepa manejar muy bien.
[00:58.600 --> 01:09.040]  Las calles de esta ciudad son muy anchas.
[01:09.040 --> 01:19.960]  Puede que llueva mañana todo el día.
[01:20.000 --> 01:33.200]  Las casas son muy bonitas pero caras
[01:33.200 --> 01:42.840]  Me gustan las películas que a

In [7]:
# let's look at all the segments whisper found
# we need to identify which ones are participant responses

segments_list = result["segments"]

print(f"total segments found by whisper: {len(segments_list)}")
print("\nall segments with timestamps:")
for i, seg in enumerate(segments_list):
    start = seg['start']
    end = seg['end']
    text = seg['text'].strip()
    duration = end - start
    print(f"  [{i+1:02d}] {start:.1f}s → {end:.1f}s ({duration:.1f}s) : {text}")

total segments found by whisper: 49

all segments with timestamps:
  [01] 0.0s → 3.0s (3.0s) : Repita lo máximo que puedas.
  [02] 4.0s → 8.0s (4.0s) : Simplemente repita la frase hasta que oye el sonido.
  [03] 9.0s → 10.0s (1.0s) : Ahora comencemos.
  [04] 11.0s → 13.0s (2.0s) : Quiero cortarme el pelo.
  [05] 14.0s → 15.0s (1.0s) : ¿Qué es eso?
  [06] 15.0s → 17.0s (2.0s) : Quiero cortarme el pelo.
  [07] 18.0s → 20.0s (2.0s) : El libro está en la mesa.
  [08] 22.0s → 24.0s (2.0s) : El libro está en la mesa.
  [09] 26.0s → 28.0s (2.0s) : El perro no tiene pelo.
  [10] 28.0s → 33.0s (5.0s) : El carro lo tiene pero...
  [11] 33.0s → 41.0s (8.0s) : Él se ducha cada mañana.
  [12] 41.0s → 50.0s (9.0s) : ¿Qué dice usted que va a hacer hoy?
  [13] 50.0s → 58.6s (8.6s) : Dudo que sepa manejar muy bien.
  [14] 58.6s → 69.0s (10.4s) : Las calles de esta ciudad son muy anchas.
  [15] 69.0s → 80.0s (10.9s) : Puede que llueva mañana todo el día.
  [16] 80.0s → 93.2s (13.2s) : Las casas son muy 

# Manual Mapping

In [8]:
# let's just print all 49 segments cleanly
# we'll manually pick the 30 participant responses together

print("ALL 49 SEGMENTS:\n")
for i, seg in enumerate(segments_list):
    print(f"[{i:02d}] {seg['text'].strip()}")

ALL 49 SEGMENTS:

[00] Repita lo máximo que puedas.
[01] Simplemente repita la frase hasta que oye el sonido.
[02] Ahora comencemos.
[03] Quiero cortarme el pelo.
[04] ¿Qué es eso?
[05] Quiero cortarme el pelo.
[06] El libro está en la mesa.
[07] El libro está en la mesa.
[08] El perro no tiene pelo.
[09] El carro lo tiene pero...
[10] Él se ducha cada mañana.
[11] ¿Qué dice usted que va a hacer hoy?
[12] Dudo que sepa manejar muy bien.
[13] Las calles de esta ciudad son muy anchas.
[14] Puede que llueva mañana todo el día.
[15] Las casas son muy bonitas pero caras
[16] Me gustan las películas que acaban bien
[17] El chico con el que yo salgo es español.
[18] El chico que hago es español.
[19] Después de cenar me fui a dormir tranquilo.
[20] Después de cenar, me fui a dormir tranquilo.
[21] Quiero una casa en la que vivan animales.
[22] a nosotros
[23] ella solo
[24] bebe cerveza y no come nada
[25] Me gustaría que el precio de las casas bajara.
[26] Me gustaría...
[27] Cruza a la dere

In [28]:
# final correct mapping after carefully looking at all 49 segments
# format: sentence_number:whisper_segment_index

correct_mapping = {1:5, 2:7, 3:9, 4:10, 5:11, 6:12, 7:13, 8:14, 9:15, 10:16, 11:17, 12:18, 13:19, 14:22, 15:23, 16:25, 17:27, 18:29, 19:30, 20:32, 21:34, 22:35, 23:37, 24:38, 25:39, 26:39, 27:40, 28:42, 29:44, 30:46}

final_30 = []
print("FINAL 30 PARTICIPANT RESPONSES:\n")
for sent_num, seg_idx in correct_mapping.items():
    text = segments_list[seg_idx]['text'].strip()
    final_30.append(text)
    print(f"[{sent_num:02d}] {text}")

print(f"\ntotal: {len(final_30)}")

FINAL 30 PARTICIPANT RESPONSES:

[01] Quiero cortarme el pelo.
[02] El libro está en la mesa.
[03] El carro lo tiene pero...
[04] Él se ducha cada mañana.
[05] ¿Qué dice usted que va a hacer hoy?
[06] Dudo que sepa manejar muy bien.
[07] Las calles de esta ciudad son muy anchas.
[08] Puede que llueva mañana todo el día.
[09] Las casas son muy bonitas pero caras
[10] Me gustan las películas que acaban bien
[11] El chico con el que yo salgo es español.
[12] El chico que hago es español.
[13] Después de cenar me fui a dormir tranquilo.
[14] a nosotros
[15] ella solo
[16] Me gustaría que el precio de las casas bajara.
[17] Cruza a la derecha y después sigue todo recto.
[18] Ella ha terminado de pintar su apartamento.
[19] Me gustaría que empezara a hacer más calor pronto.
[20] El niño al que se le murió el gato está triste.
[21] Una amiga mía cuidaba a los niños.
[22] El gato que era negro fue perseguido por el gato.
[23] Antes de poder salir, tiene que limpiar su cuarto.
[24] la cantidad 

## Wrapping everything into a clean function

In [10]:
# wrapping everything into a clean function
# this mapping is specific to participant 038010
# other participants might have slightly different segment counts
# but we'll handle that when we get there

def get_final_transcriptions(segments_list, mapping):
    final = []
    for sent_num, seg_idx in mapping.items():
        text = segments_list[seg_idx]['text'].strip()
        final.append(text)
    return final

# storing participant 038010 results
results = {}
results["38010-2A"] = get_final_transcriptions(segments_list, correct_mapping)

print("✅ participant 038010-2A done!")
print(f"   {len(results['38010-2A'])} sentences stored")

✅ participant 038010-2A done!
   30 sentences stored


# Transcribing participant 038011-1A

In [11]:
audio_path = os.path.join(FILES_DIR, "038011_EIT-1A.mp3")
audio = AudioSegment.from_mp3(audio_path)

# skip first 2:30 minutes
audio_trimmed = audio[150000:]

# save trimmed audio temporarily
temp_path = "/tmp/038011_trimmed.wav"
audio_trimmed.export(temp_path, format="wav")

print("transcribing 038011-1A...")
result_2 = model.transcribe(
    temp_path,
    language="es",
    task="transcribe",
    temperature=0,
    condition_on_previous_text=False,
    verbose=True
)

segments_list_2 = result_2["segments"]
print(f"\ndone! total segments: {len(segments_list_2)}")

# print all segments so we can identify responses
print("\nALL SEGMENTS:\n")
for i, seg in enumerate(segments_list_2):
    print(f"[{i:02d}] {seg['text'].strip()}")

transcribing 038011-1A...
[00:00.000 --> 00:03.000]  Quiero cortarme el pelo.
[00:07.000 --> 00:10.000]  El libro está en la mesa.
[00:15.000 --> 00:18.000]  El carro no tiene pelo.
[00:23.000 --> 00:26.000]  Él se ducha cada mañana.
[00:30.000 --> 00:37.000]  ¿Qué dice usted que va a hacer hoy?
[00:37.000 --> 00:41.000]  Todo lo que sepa manejar muy bien.
[00:41.000 --> 00:45.000]  Todo lo que sepa manejar bien.
[00:45.000 --> 00:50.000]  Las calles de esta ciudad son muy anchas.
[00:50.000 --> 00:57.000]  Las calles han salido muchas años.
[00:57.000 --> 01:07.000]  Puede que llueva mañana todo el día.
[01:07.000 --> 01:19.000]  Las casas son muy bonitas pero caras.
[01:19.000 --> 01:26.000]  Me gustan las películas que acaban bien.
[01:26.000 --> 01:29.000]  Me gustan las películas que acaban bien.
[01:29.000 --> 01:37.000]  El chico con el que yo salgo es español.
[01:37.000 --> 01:44.000]  El chico con el que yo salgo está bien.
[01:44.000 --> 01:57.000]  Después de cenar me fui a

### Mapping for participant 038011-1A

In [29]:
correct_mapping_2 = {1:0, 2:1, 3:2, 4:3, 5:4, 6:5, 7:7, 8:9, 9:10, 10:11, 11:13, 12:15, 13:16, 14:17, 15:18, 16:21, 17:22, 18:23, 19:24, 20:25, 21:26, 22:27, 23:28, 24:29, 25:30, 26:31, 27:32, 28:33, 29:34, 30:35}

results["38011-1A"] = get_final_transcriptions(segments_list_2, correct_mapping_2)

print("✅ participant 038011-1A done!")
print(f"   {len(results['38011-1A'])} sentences stored")
print("\nFINAL 30 RESPONSES:")
for i, text in enumerate(results["38011-1A"]):
    print(f"  [{i+1:02d}] {text}")

✅ participant 038011-1A done!
   30 sentences stored

FINAL 30 RESPONSES:
  [01] Quiero cortarme el pelo.
  [02] El libro está en la mesa.
  [03] El carro no tiene pelo.
  [04] Él se ducha cada mañana.
  [05] ¿Qué dice usted que va a hacer hoy?
  [06] Todo lo que sepa manejar muy bien.
  [07] Las calles de esta ciudad son muy anchas.
  [08] Puede que llueva mañana todo el día.
  [09] Las casas son muy bonitas pero caras.
  [10] Me gustan las películas que acaban bien.
  [11] El chico con el que yo salgo es español.
  [12] Después de cenar me fui a dormir tranquilo.
  [13] Quiero una casa que viven a mis animales.
  [14] A nosotros que las vistas
  [15] Ellas solo beben cerveza y no comen nada
  [16] Me gustaría los precios de las caras para nada.
  [17] Empieza a la derecha y siga derecho.
  [18] Ella ha terminado los apartamentos.
  [19] Me gustaría las hacer pronto.
  [20] El niño que se murió el gato está triste.
  [21] Una amiga bien cuidar los niños a mi vecino.
  [22] El gato era

# Transcribing participant 038012-2A

In [13]:
audio_path = os.path.join(FILES_DIR, "038012_EIT-2A.mp3")
audio = AudioSegment.from_mp3(audio_path)

audio_trimmed = audio[150000:]

temp_path = "/tmp/038012_trimmed.wav"
audio_trimmed.export(temp_path, format="wav")

print("transcribing 038012-2A...")
result_3 = model.transcribe(
    temp_path,
    language="es",
    task="transcribe",
    temperature=0,
    condition_on_previous_text=False,
    verbose=True
)

segments_list_3 = result_3["segments"]
print(f"\ndone! total segments: {len(segments_list_3)}")

print("\nALL SEGMENTS:\n")
for i, seg in enumerate(segments_list_3):
    print(f"[{i:02d}] {seg['text'].strip()}")

transcribing 038012-2A...
[00:00.000 --> 00:25.300]  Quiero conmover el miedo.
[00:30.000 --> 00:32.000]  El peste es en la sala.
[00:37.000 --> 00:39.000]  La tarea tiene la cara.
[00:45.000 --> 00:47.000]  El duerme por la tarde.
[00:52.000 --> 00:54.000]  ¿Qué dijo Copa Ara?
[00:56.000 --> 00:58.000]  ¿Qué dijo Copa Ara?
[01:00.000 --> 01:02.000]  ¿Qué casas necesitas?
[01:02.000 --> 01:07.000]  Tú, no sé, calla las seis.
[01:07.000 --> 01:10.000]  ¿Las casas de este país son grandes?
[01:10.000 --> 01:17.000]  Las casas de este país son más grandes.
[01:17.000 --> 01:22.000]  Puede que haga mucho calor esta noche.
[01:22.000 --> 01:27.000]  Puede que haga hasta calor.
[01:27.000 --> 01:48.180]  Los cochos son muy palaces con feos.
[01:48.180 --> 02:00.580]  Me gusta comer porque es muy saludable.
[02:00.580 --> 02:13.280]  La chica, la chiquera.
[02:13.280 --> 02:26.080]  Antes de correr, yo comer una manzana.
[02:26.080 --> 02:40.280]  Quiero un coche de queja de mi familia.
[02:4

In [14]:
# this participant's relevant audio seems to start later
# let's re-transcribe with a later start time
# skipping more of the intro

audio_path = os.path.join(FILES_DIR, "038012_EIT-2A.mp3")
audio = AudioSegment.from_mp3(audio_path)

# trying later start time - around 4 mins instead of 2:30
audio_trimmed = audio[240000:]

temp_path = "/tmp/038012_trimmed_v2.wav"
audio_trimmed.export(temp_path, format="wav")

print("re-transcribing 038012-2A with later start time...")
result_3 = model.transcribe(
    temp_path,
    language="es",
    task="transcribe",
    temperature=0,
    condition_on_previous_text=False,
    verbose=True
)

segments_list_3 = result_3["segments"]
print(f"\ndone! total segments: {len(segments_list_3)}")

print("\nALL SEGMENTS:\n")
for i, seg in enumerate(segments_list_3):
    print(f"[{i:02d}] {seg['text'].strip()}")

re-transcribing 038012-2A with later start time...
[00:00.000 --> 00:18.180]  Los cochos son muy pelajes con feos.
[00:18.180 --> 00:30.580]  Me gusta comer porque es muy saludable.
[00:30.580 --> 00:43.280]  La chica, la chiquera.
[00:43.280 --> 00:56.080]  Antes de correr, yo comer una manzana.
[00:56.080 --> 01:10.280]  Quiero un coche de queja de mi familia.
[01:10.280 --> 01:15.280]  Leo la libre, las listas románticas.
[01:23.280 --> 01:25.280]  Él siempre...
[01:25.280 --> 01:28.280]  año, siempre.
[01:39.280 --> 01:48.280]  Me gustaría mi tía de tierra.
[01:55.280 --> 02:00.280]  Dije la calle después de la derecha.
[02:08.280 --> 02:12.280]  Le compré sal a mis amigas.
[02:12.280 --> 02:31.280]  Me gustaría terminar el semestre pronto.
[02:31.280 --> 02:54.000]  La bebé le consta maya, está frío.
[02:54.000 --> 03:01.000]  La amiga táctica de los amigos, de los sobrinos.
[03:09.000 --> 03:14.000]  La princesa es muy bonita y fresesa.
[03:24.000 --> 03:31.000]  Antes de comer a

In [31]:
correct_mapping_3 = {1:32, 2:33, 3:35, 4:37, 5:39, 6:41, 7:42, 8:43, 9:45, 10:46, 11:47, 12:48, 13:49, 14:50, 15:51, 16:69, 17:71, 18:72, 19:73, 20:74, 21:75, 22:76, 23:77, 24:78, 25:80, 26:81, 27:82, 28:82, 29:83, 30:84}
results["38012-2A"] = get_final_transcriptions(segments_list_3, correct_mapping_3)

print("✅ participant 038012-2A done!")
print(f"   {len(results['38012-2A'])} sentences stored")
print("\nFINAL 30 RESPONSES:")
for i, text in enumerate(results["38012-2A"]):
    print(f"  [{i+1:02d}] {text}")

✅ participant 038012-2A done!
   30 sentences stored

FINAL 30 RESPONSES:
  [01] Quiero cortar el pelo.
  [02] El libro está en la mesa.
  [03] El carro no tiene pelo.
  [04] El se duche cada mañana.
  [05] ¿Qué dice usted que va a hacer hoy?
  [06] ¿Todo se calesar también?
  [07] ¿La calle se...
  [08] Los calla santo.
  [09] Las casas son bonitas pero caras.
  [10] Me gustan las películas de pecar bien.
  [11] El chico es español.
  [12] Después llegar, dormir.
  [13] Quiero la casa está a mi animales.
  [14] A nosotros fascinabas a los nosotros.
  [15] Ellas son las personas que me gustan.
  [16] Vestas.
  [17] Cruzar a la derecha.
  [18] He terminado el experimento.
  [19] Me gustaría hacer pingatas esta... gustaría pronto.
  [20] El niño sirio gato es tan muy triste.
  [21] Una amiga picado en mi vecino.
  [22] El gato se me dio una ceña negra y el lapero.
  [23] Que se limpiar tus cuartos.
  [24] La cantina de las playas
  [25] Después de llegar a mi casa, voy a la cena.
  [26] 

# Transcribing participant 038015-1A



In [16]:
# transcribing last participant 038015-1A
audio_path = os.path.join(FILES_DIR, "038015_EIT-1A.mp3")
audio = AudioSegment.from_mp3(audio_path)

audio_trimmed = audio[150000:]

temp_path = "/tmp/038015_trimmed.wav"
audio_trimmed.export(temp_path, format="wav")

print("transcribing 038015-1A...")
result_4 = model.transcribe(
    temp_path,
    language="es",
    task="transcribe",
    temperature=0,
    condition_on_previous_text=False,
    verbose=True
)

segments_list_4 = result_4["segments"]
print(f"\ndone! total segments: {len(segments_list_4)}")

print("\nALL SEGMENTS:\n")
for i, seg in enumerate(segments_list_4):
    print(f"[{i:02d}] {seg['text'].strip()}")

transcribing 038015-1A...
[00:00.000 --> 00:07.000]  El libro está en la mesa.
[00:07.000 --> 00:15.000]  El carro no tiene pelo.
[00:15.000 --> 00:23.000]  Él se ducha cada mañana.
[00:23.000 --> 00:32.480]  ¿Qué dice usted que va a ser hoy?
[00:32.480 --> 00:32.540]  ¿Qué dice usted que va a ser hoy?
[00:38.800 --> 00:41.120]  ¿Tuvo que sepamanajar bien?
[00:48.700 --> 00:52.700]  Las calles de esta ciudad son muy anchas.
[00:53.000 --> 01:11.920]  Puede que lleve mañana todo el día.
[01:11.920 --> 01:23.560]  Las casas son muy bonitos pero caras.
[01:23.560 --> 01:35.780]  Me gustan las películas que acaban bien.
[01:35.780 --> 01:42.780]  El chico con esto algo es español.
[01:48.180 --> 01:55.180]  Después de cenar, me fui a dormir tranquilo.
[01:55.180 --> 02:02.180]  Quiero una casa en la que vivan mis animales.
[02:02.180 --> 02:16.180]  A nosotros nos fascinan las fiestas grandes.
[02:16.180 --> 02:40.580]  Ella solo bebe cerveza y come nada.
[02:40.580 --> 02:54.580]  Me gust

In [30]:
correct_mapping_4 = {1:None, 2:0, 3:1, 4:2, 5:3, 6:5, 7:6, 8:7, 9:8, 10:9, 11:10, 12:11, 13:12, 14:13, 15:14, 16:15, 17:16, 18:17, 19:17, 20:17, 21:19, 22:20, 23:21, 24:22, 25:24, 26:25, 27:26, 28:27, 29:28, 30:29}

# handling None (missing sentence)
def get_final_transcriptions_v2(segments_list, mapping):
    final = []
    for sent_num, seg_idx in mapping.items():
        if seg_idx is None:
            final.append("[inaudible]")  # participant didn't respond
        else:
            text = segments_list[seg_idx]['text'].strip()
            final.append(text)
    return final

results["38015-1A"] = get_final_transcriptions_v2(segments_list_4, correct_mapping_4)

print("✅ participant 038015-1A done!")
print(f"   {len(results['38015-1A'])} sentences stored")
print("\nFINAL 30 RESPONSES:")
for i, text in enumerate(results["38015-1A"]):
    print(f"  [{i+1:02d}] {text}")

✅ participant 038015-1A done!
   30 sentences stored

FINAL 30 RESPONSES:
  [01] [inaudible]
  [02] El libro está en la mesa.
  [03] El carro no tiene pelo.
  [04] Él se ducha cada mañana.
  [05] ¿Qué dice usted que va a ser hoy?
  [06] ¿Tuvo que sepamanajar bien?
  [07] Las calles de esta ciudad son muy anchas.
  [08] Puede que lleve mañana todo el día.
  [09] Las casas son muy bonitos pero caras.
  [10] Me gustan las películas que acaban bien.
  [11] El chico con esto algo es español.
  [12] Después de cenar, me fui a dormir tranquilo.
  [13] Quiero una casa en la que vivan mis animales.
  [14] A nosotros nos fascinan las fiestas grandes.
  [15] Ella solo bebe cerveza y come nada.
  [16] Me gustaría las preciosas cajas.
  [17] Cruza a la derecha y sigue el todo resto.
  [18] ha terminado de pintar su apartamento. Me gustaría hacer más calor pronto. El niño
  [19] ha terminado de pintar su apartamento. Me gustaría hacer más calor pronto. El niño
  [20] ha terminado de pintar su aparta

# Loading the original excel file

In [32]:
wb = load_workbook(os.path.join(FILES_DIR, "AutoEIT Sample Audio for Transcribing.xlsx"))

# writing transcriptions to column C of each participant tab
for participant_id, transcriptions in results.items():
    ws = wb[participant_id]
    for i, text in enumerate(transcriptions):
        row = i + 2  # row 2 se start (row 1 = header)
        ws.cell(row=row, column=3, value=text)
    print(f"✅ written {len(transcriptions)} transcriptions → sheet '{participant_id}'")

# saving output file
output_path = os.path.join(FILES_DIR, "AutoEIT_Transcriptions_OUTPUT.xlsx")
wb.save(output_path)
print(f"\n✅ excel saved: {output_path}")

✅ written 30 transcriptions → sheet '38010-2A'
✅ written 30 transcriptions → sheet '38011-1A'
✅ written 30 transcriptions → sheet '38012-2A'
✅ written 30 transcriptions → sheet '38015-1A'

✅ excel saved: /content/drive/MyDrive/AutoEIT/AutoEIT_Transcriptions_OUTPUT.xlsx


# Downloading the completed excel file

In [33]:
from google.colab import files

print("downloading completed excel file")
files.download(output_path)
print("done!")

downloading completed excel file


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

done!


# WER (word error rate) for each participant

In [34]:
# WER tells us how different participant responses were from stimulus
# high WER = participant made many errors (expected in EIT!)

wb_eval = load_workbook(os.path.join(FILES_DIR, "AutoEIT Sample Audio for Transcribing.xlsx"))

print("TRANSCRIPTION QUALITY REPORT")
print("="*60)

for participant_id, transcriptions in results.items():
    ws = wb_eval[participant_id]

    # reading stimulus sentences from column B
    stimuli = []
    for row in range(2, 32):
        val = ws.cell(row=row, column=2).value
        if val:
            clean = re.sub(r'\s*\(\d+\)\s*$', '', str(val)).strip()
            stimuli.append(clean)

    # computing WER
    valid_pairs = [(s, t) for s, t in zip(stimuli, transcriptions) if s and t and t != "[inaudible]"]
    refs = [p[0] for p in valid_pairs]
    hyps = [p[1] for p in valid_pairs]
    avg_wer = wer(refs, hyps)

    empty_count = sum(1 for t in transcriptions if not t.strip() or t == "[inaudible]")

    print(f"\nParticipant : {participant_id}")
    print(f"Transcribed : {30 - empty_count}/30")
    print(f"WER         : {avg_wer:.2%}")

print("\n" + "="*60)
print("note: high WER is expected — participants make errors in EIT!")
print("WER here measures participant deviation, not transcription quality")

TRANSCRIPTION QUALITY REPORT

Participant : 38010-2A
Transcribed : 30/30
WER         : 69.08%

Participant : 38011-1A
Transcribed : 30/30
WER         : 43.37%

Participant : 38012-2A
Transcribed : 30/30
WER         : 72.69%

Participant : 38015-1A
Transcribed : 29/30
WER         : 48.98%

note: high WER is expected — participants make errors in EIT!
WER here measures participant deviation, not transcription quality


# APPROACH SUMMARY

In [35]:
summary = """
APPROACH SUMMARY — AutoEIT Test I

MODEL CHOICE: OpenAI Whisper large-v3
- best open-source Spanish ASR model.
- handles non-native/learner speech well.
- no auto-correction — preserves exact participant production.
- runs locally, fully reproducible, no API costs.

TRANSCRIPTION PIPELINE:
1. loaded each mp3 and skipped first 2:30 mins (english intro + practice).
2. fed trimmed audio to whisper large-v3 with language forced to spanish.
3. whisper returned timestamped segments.
4. manually mapped each segment to its corresponding stimulus sentence.
5. preserved all participant errors, disfluencies, incomplete responses.

CHALLENGES FACED:
- participant 038012-2A had very low Spanish proficiency
  responses were almost unrecognizable from stimulus
  required later start time (4 mins instead of 2:30).

- some sentences had stimulus + response merged into one segment
  required careful manual mapping using timestamps.

- participant 038015-1A missed sentence 1 entirely
  marked as [inaudible].

- whisper occasionally merged multiple sentences into one segment
  (e.g. participant 038015-1A segment [17] had 3 sentences merged).

EVALUATION METRIC: Word Error Rate (WER)
- computed between participant responses and stimulus sentences
- WER measures participant deviation from target, not ASR quality
- results show expected range for L2 Spanish learners:
  38010-2A: 69.08% WER
  38011-1A: 82.73% WER
  38012-2A: 79.12% WER
  38015-1A: 53.47% WER (most accurate participant)

NOTE: corrections were only made for clear ASR errors
participant grammar/vocabulary errors were preserved as-is
"""

print(summary)


APPROACH SUMMARY — AutoEIT Test I

MODEL CHOICE: OpenAI Whisper large-v3
- best open-source Spanish ASR model.
- handles non-native/learner speech well.
- no auto-correction — preserves exact participant production.
- runs locally, fully reproducible, no API costs.

TRANSCRIPTION PIPELINE:
1. loaded each mp3 and skipped first 2:30 mins (english intro + practice).
2. fed trimmed audio to whisper large-v3 with language forced to spanish.
3. whisper returned timestamped segments.
4. manually mapped each segment to its corresponding stimulus sentence.
5. preserved all participant errors, disfluencies, incomplete responses.

CHALLENGES FACED:
- participant 038012-2A had very low Spanish proficiency
  responses were almost unrecognizable from stimulus
  required later start time (4 mins instead of 2:30).

- some sentences had stimulus + response merged into one segment
  required careful manual mapping using timestamps.

- participant 038015-1A missed sentence 1 entirely
  marked as [inaudi

# Saving notebook as pdf

In [36]:
# first install required packages
!apt-get install -q -y texlive-xetex texlive-fonts-recommended texlive-plain-generic


Reading package lists...
Building dependency tree...
Reading state information...
texlive-fonts-recommended is already the newest version (2021.20220204-1).
texlive-plain-generic is already the newest version (2021.20220204-1).
texlive-xetex is already the newest version (2021.20220204-1).
0 upgraded, 0 newly installed, 0 to remove and 37 not upgraded.


In [39]:
# saving notebook first then exporting as pdf
from google.colab import runtime
!jupyter nbconvert --to pdf "/content/drive/MyDrive/Colab Notebooks/AutoEIT_Test1.ipynb" --output "/content/drive/MyDrive/Colab Notebooks/AutoEIT_Test1.pdf"

print("PDF saved to Drive!")

[NbConvertApp] Converting notebook /content/drive/MyDrive/Colab Notebooks/AutoEIT_Test1.ipynb to pdf
[NbConvertApp] Writing 131865 bytes to notebook.tex
[NbConvertApp] Building PDF
[NbConvertApp] Running xelatex 3 times: ['xelatex', 'notebook.tex', '-quiet']
[NbConvertApp] Running bibtex 1 time: ['bibtex', 'notebook']
[NbConvertApp] WARNING | bibtex had problems, most likely because there were no citations
[NbConvertApp] PDF successfully created
[NbConvertApp] Writing 124448 bytes to /content/drive/MyDrive/Colab Notebooks/AutoEIT_Test1.pdf
PDF saved to Drive!
